# CCS4354 – Tensors and Graphs
# Graph Neural Networks for Node Classification on the OGBN-Arxiv Citation Network

**Dataset:** OGBN-Arxiv  

**Group members**

- CIT-23-02-0162 – Jayani Sashikala
- CIT-23-02-0130 – Sandani Senevithna
- CIT-23-02-0073 – Ganguli Kaluarachchi
- CIT-23-02-0358 – Dureksha Arangala

## Notebook Structure

| Section | Required work |
|---|---|
| Task 01 | PyTorch tensor operations |
| Task 02 | Graph representation and structural analysis |
| Task 03 | Features, labels, splits, and preprocessing |
| Task 04 | GCN and GraphSAGE implementation |
| Task 05 | Training, hyperparameter tuning, and monitoring |
| Task 06 | Validation and test evaluation |
| Task 07 | PCA embeddings and neighbourhood influence |
| Task 08 | Streamlit graph-intelligence dashboard |

## Shared Setup

### Minimal Shared Setup

Only file paths are prepared here. Every task imports its own relevant libraries beside the code that uses them.

In [ ]:
!pip install -q torch-geometric ogb streamlit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 43.0 MB/s eta 0:00:00


### Project Paths and Output Folders

This section defines a single project root and creates consistent locations for datasets, trained models, figures, tables, and dashboard files.

In [ ]:
import os
# Compatibility fix for trusted PyG objects downloaded by the official OGB package on PyTorch 2.6+.
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
PROJECT_ROOT = Path('/content/drive/MyDrive/OGBN_Arxiv_Project')
RESULTS_ROOT = PROJECT_ROOT / 'results'
MODELS_DIR = PROJECT_ROOT / 'models'
SRC_DIR = PROJECT_ROOT / 'src'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'

for folder in [PROJECT_ROOT, RESULTS_ROOT, MODELS_DIR, SRC_DIR, ARTIFACTS_DIR,
               PROJECT_ROOT / 'notebooks', PROJECT_ROOT / 'visualizations',
               PROJECT_ROOT / 'dashboard', PROJECT_ROOT / 'report',
               PROJECT_ROOT / 'presentation', PROJECT_ROOT / 'video']:
    folder.mkdir(parents=True, exist_ok=True)
print('Project root:', PROJECT_ROOT)

Mounted at /content/drive
Project root: /content/drive/MyDrive/OGBN_Arxiv_Project


### Imports, Reproducibility, and Computing Device

The imports support the complete workflow. Fixed random seeds make experiments more reproducible, while device selection allows the same notebook to use a GPU when one is available.

In [ ]:
# Common reproducibility settings used by later tasks
import random
import warnings
import numpy as np

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

### OGBN-Arxiv Dataset and Shared Graph Variables

The dataset is loaded once and its node features, labels, citation edges, official data splits, and basic dimensions are stored for reuse throughout the notebook.

# Task 01 – Tensor Fundamentals

Required PyTorch tensor concepts and GPU operation demonstration.

## 1.1 Tensor Creation

Creates tensors with different values, shapes, and data types to demonstrate the basic data structure used by PyTorch.

In [ ]:
# Libraries required for Task 01
import torch

torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Tensor creation
one_dimensional = torch.tensor([1, 2, 3, 4])
two_dimensional = torch.tensor([[1, 2], [3, 4]])
matrix = torch.tensor([[1, 2, 3], [4, 5, 6]])
zeros = torch.zeros((2, 3))
random_tensor = torch.rand((2, 3))

print('1D tensor:', one_dimensional)
print('2D tensor:\n', two_dimensional)
print('Zeros:\n', zeros)
print('Random tensor:\n', random_tensor)
print('Device:', DEVICE)

1D tensor: tensor([1, 2, 3, 4])
2D tensor:
 tensor([[1, 2],
        [3, 4]])
Zeros:
 tensor([[0., 0., 0.],
        [0., 0., 0.]])
Random tensor:
 tensor([[0.8823, 0.9150, 0.3829],
        [0.9593, 0.3904, 0.6009]])
Device: cpu


## 1.2 Tensor Indexing and Reshaping

Shows how selected values are accessed and how tensor dimensions are reorganized without changing the underlying information.

In [ ]:
# Indexing
print('First row:', matrix[0])
print('Second row, third value:', matrix[1, 2])
print('Second column:', matrix[:, 1])

# Reshaping
values = torch.arange(12)
reshaped = values.reshape(3, 4)
print('\nReshaped tensor:\n', reshaped)
print('Flattened:', reshaped.flatten())

First row: tensor([1, 2, 3])
Second row, third value: tensor(6)
Second column: tensor([2, 5])

Reshaped tensor:
 tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
Flattened: tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])


## 1.3 Matrix Multiplication and Broadcasting

Demonstrates feature transformation through matrix multiplication and automatic expansion of compatible tensor shapes through broadcasting.

In [ ]:
# Matrix multiplication
A = torch.tensor([[1., 2.], [3., 4.]])
B = torch.tensor([[5., 6.], [7., 8.]])
print('Matrix multiplication:\n', A @ B)

# Broadcasting: the vector is added to every row
features = torch.tensor([[1., 2., 3.], [4., 5., 6.]])
bias = torch.tensor([10., 20., 30.])
print('\nBroadcasting result:\n', features + bias)

Matrix multiplication:
 tensor([[19., 22.],
        [43., 50.]])

Broadcasting result:
 tensor([[11., 22., 33.],
        [14., 25., 36.]])


## 1.4 Aggregation and GPU Tensor Operations

Calculates summary values such as sums and means, then demonstrates how tensors and calculations can be placed on the selected computing device.

In [ ]:
# Aggregation
print('Sum:', features.sum())
print('Mean:', features.mean())
print('Column sums:', features.sum(dim=0))
print('Row means:', features.mean(dim=1))

# GPU operation if CUDA is available
gpu_tensor = torch.rand((100, 100)).to(DEVICE)
gpu_result = gpu_tensor @ gpu_tensor
print('Tensor device:', gpu_result.device)
print('Result shape:', gpu_result.shape)

Sum: tensor(21.)
Mean: tensor(3.5000)
Column sums: tensor([5., 7., 9.])
Row means: tensor([2., 5.])
Tensor device: cpu
Result shape: torch.Size([100, 100])


## 1.5 Interpretation

Indexing selects data splits, reshaping prepares labels, matrix multiplication transforms features, broadcasting applies compatible values across rows, and aggregation combines neighbour information. The same operations form the numerical foundation of graph neural networks.